In [ ]:
from google.colab import auth
auth.authenticate_user()

!gcloud config set project master-thesis-measles -q

INFORMATION: Project 'master-thesis-measles' has no 'environment' tag set. Use either 'Production', 'Development', 'Test', or 'Staging'. Add an 'environment' tag using `gcloud resource-manager tags bindings create`.
Updated property [core/project].


In [ ]:
# === Export all TFRecords to GCS (quiet mode, wealthpooled) ===
import ee, math, time, pandas as pd
from typing import Dict, Tuple, Any
from tqdm.auto import tqdm

ee.Authenticate()
ee.Initialize(project='master-thesis-measles')

# --- CONFIG ---
CSV_PATH     = "/content/drive/My Drive/Master_Thesis/Surveys/clusters_yeh_spec.csv"
TFRECORD_DIR = "gs://ssadata"     # use exactly this
SCALE        = 30
EXPORT_TILE_RADIUS = 127
CHUNK_SIZE   = 20                  # households per TFRecord shard
BANDS = ['BLUE','GREEN','RED','NIR','SWIR1','SWIR2','TEMP1','LON','LAT','NIGHTLIGHTS']

# --- parse bucket name from TFRECORD_DIR ---
GCS_BUCKET = TFRECORD_DIR.replace("gs://", "").split("/")[0]

def three_year_window(y: int) -> Tuple[str, str]:
    if   2015 <= y <= 2017: return "2015-01-01", "2017-12-31"
    elif 2018 <= y <= 2020: return "2018-01-01", "2020-12-31"
    elif 2021 <= y <= 2023: return "2021-01-01", "2023-12-31"
    elif 2024 <= y <= 2026: return "2024-01-01", "2026-12-31"
    else:                   return f"{y-1}-01-01", f"{y+1}-12-31"

def mask_c2_qa(img):
    qa = img.select('QA_PIXEL')
    keep = (qa.bitwiseAnd(1 << 3).eq(0)
            .And(qa.bitwiseAnd(1 << 4).eq(0))
            .And(qa.bitwiseAnd(1 << 5).eq(0))
            .And(qa.bitwiseAnd(1 << 7).eq(0)))
    return img.updateMask(keep)

def landsat_stack(roi, start, end):
    sel = ['SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7','ST_B10']
    l8 = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
          .filterBounds(roi).filterDate(start, end)
          .map(mask_c2_qa).select(sel))
    l9 = (ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
          .filterBounds(roi).filterDate(start, end)
          .map(mask_c2_qa).select(sel))
    return (l8.merge(l9)).median().rename(
        ['BLUE','GREEN','RED','NIR','SWIR1','SWIR2','TEMP1']
    )

def composite_viirs(start, end):
    return (ee.ImageCollection('NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG')
            .filterDate(start, end).median()
            .select(['avg_rad'], ['NIGHTLIGHTS']))

def add_lonlat(img):
    latlon = ee.Image.pixelLonLat().select(['longitude','latitude'], ['LON','LAT'])
    return img.addBands(latlon)

def df_to_features(df):
    feats = []
    for _, r in df.iterrows():
        geom = ee.Geometry.Point([float(r['lon']), float(r['lat'])])
        feats.append(ee.Feature(geom, {
            'country':        str(r['country']),
            'year':           int(r['year']),
            'cluster_index':  int(r['cluster']),
            'lon':            float(r['lon']),
            'wealthpooled':   float(r['wealthpooled']),   # <- use wealthpooled directly
        }))
    return ee.FeatureCollection(feats)

def make_array_image(img, radius_px):
    kern = ee.Kernel.square(radius=radius_px, units='pixels')
    arr_imgs = [img.select([b]).neighborhoodToArray(kern).rename(b) for b in BANDS]
    return ee.Image.cat(arr_imgs)

def export_chunk(yr, base_img, points_fc, fname):
    start, end = three_year_window(yr)
    img = add_lonlat(base_img).addBands(composite_viirs(start, end)).select(BANDS)
    arr_img = make_array_image(img, EXPORT_TILE_RADIUS)

    def _sample_point(f):
        s = (arr_img.sample(region=f.geometry(),
                            scale=SCALE,
                            projection='EPSG:3857',
                            numPixels=1,
                            dropNulls=False,
                            tileScale=12).first())
        out = ee.Feature(None).copyProperties(f, ['country','year','cluster_index','lon','wealthpooled'])
        return out.setMulti(s.toDictionary(ee.List(BANDS)))

    samples = points_fc.map(_sample_point)

    task = ee.batch.Export.table.toCloudStorage(
        collection=samples,
        description=fname,
        bucket=GCS_BUCKET,             # bucket name only
        fileNamePrefix=fname,          # object prefix in the bucket
        fileFormat='TFRecord',
        selectors=['country','year','cluster_index','lon','wealthpooled'] + BANDS
    )
    task.start()
    return task

def run_exports():
    df = pd.read_csv(CSV_PATH)
    # Require wealthpooled instead of mean_pca_wealth
    need = {'country','year','cluster','lat','lon','wealthpooled'}
    missing = need - set(df.columns)
    if missing:
        raise ValueError(f"{CSV_PATH} missing columns: {sorted(missing)}")

    df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
    df = df.dropna(subset=['lat','lon','year','cluster','wealthpooled'])
    df = df.loc[df['year'] >= 2016].copy()
    if df.empty:
        raise SystemExit("No rows with year >= 2016")

    groups = list(df.groupby(['country','year']))
    n_tasks = 0

    for (cc, yr), g in tqdm(groups, desc="Exporting (country,year)"):
        g = g.reset_index(drop=True)
        roi = ee.Geometry.MultiPoint(g[['lon','lat']].values.tolist())
        start, end = three_year_window(int(yr))
        base = landsat_stack(roi, start, end)

        n = len(g)
        n_chunks = math.ceil(n / CHUNK_SIZE) if CHUNK_SIZE else 1
        for i in range(n_chunks):
            sl = slice(i*CHUNK_SIZE, min((i+1)*CHUNK_SIZE, n))
            fc = df_to_features(g.iloc[sl].copy())
            fname = f"{cc}_{int(yr)}_{i:02d}"
            export_chunk(int(yr), base, fc, fname)
            n_tasks += 1
            time.sleep(0.25)  # be polite to the API

    print(f"Started {n_tasks} export tasks to bucket '{GCS_BUCKET}'.")
    print("Monitor progress in the Earth Engine Tasks tab.")

run_exports()